# V-Mode Normalization Study (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-04-12  
**Last Modified:** 2026-04-12  
**Status:** In Progress  
**Keywords:** Rubin Observatory, AOS, normalization, V-modes, sensitivity matrix  

## Description

Focused study of how different normalization schemes affect V-mode decomposition,
using a minimal 2-DOF, 1-Zernike example to make the algebra transparent.

The goal is to understand what normalization puts all DOFs on the same footing
in the normalized sensitivity matrix, so that V-modes properly combine degenerate DOFs.

**Output:** Printed tables of sensitivity matrix values, normalization weights, and V-modes

**Based on:** smatrix_vmode_info.ipynb

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04-12 | Aaron Roodman | Initial version |

## Index Notation Conventions

| Symbol | Meaning | Range |
|--------|---------|-------|
| $i$ | Degree of Freedom (DOF) index | 0-49 (or active subset) |
| $j$ | Pupil-plane Zernike index (Noll) | 4-26 |
| $k$ | Focal-plane Zernike index (double-Zernike $k_{\text{fp}}$) | 1, 2, 3 |
| $m$ | V-mode index | 1-$N_{\text{trunc}}$ |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Build Sensitivity Matrix](#smatrix)
4. [Normalization Scheme 1: Default (r_i x f_i)](#norm1)
5. [Normalization Scheme 2: FWHM per Normalized DOF](#norm2)
6. [SVD and V-Mode Comparison](#svd)
7. [Analysis: Which Normalization Equalizes DOFs?](#analysis)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# OFC config
ofc_config_dir = '/home/r/roodman/u/LSST/packages/ts_config_mttcs/MTAOS/v13/ofc'

# Zernike selection (pupil-plane, Noll indices)
zn_select = np.array([4])   # Just focus (Z4) for simplicity

# DOF selection (indices into the 50-DOF vector)
# Default: Cam_dz (index 5) and Cam_rx (index 8)
dof_names = ['Cam_dz', 'Cam_rx']
dof_indices = [5, 8]

# DOF labels (all 50)
labels_50dof = [
    'M2_dz', 'M2_dx', 'M2_dy', 'M2_rx', 'M2_ry',
    'Cam_dz', 'Cam_dx', 'Cam_dy', 'Cam_rx', 'Cam_ry',
    'B1_1', 'B1_2', 'B1_3', 'B1_4', 'B1_5',
    'B1_6', 'B1_7', 'B1_8', 'B1_9', 'B1_10',
    'B1_11', 'B1_12', 'B1_13', 'B1_14', 'B1_15',
    'B1_16', 'B1_17', 'B1_18', 'B1_19', 'B1_20',
    'B2_1', 'B2_2', 'B2_3', 'B2_4', 'B2_5',
    'B2_6', 'B2_7', 'B2_8', 'B2_9', 'B2_10',
    'B2_11', 'B2_12', 'B2_13', 'B2_14', 'B2_15',
    'B2_16', 'B2_17', 'B2_18', 'B2_19', 'B2_20',
]

# WFS corners
sensor_name_list = ['R00_SW0', 'R04_SW0', 'R40_SW0', 'R44_SW0']
n_wfs = len(sensor_name_list)

print(f'Zernikes: {zn_select}')
print(f'DOFs: {dof_names} (indices {dof_indices})')
print(f'WFS: {sensor_name_list}')
print(f'Measurement vector: {len(zn_select)} Zernike x {n_wfs} WFS = {len(zn_select) * n_wfs} elements')
print(f'DOFs: {len(dof_indices)}')

<a id='setup'></a>
## Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lsst.ts.ofc import OFCData, SensitivityMatrix, StateEstimator
from lsst.ts.ofc import BendModeToForce
from lsst.ts.wep.utils import convertZernikesToPsfWidth

%matplotlib inline

<a id='smatrix'></a>
## 1. Build Sensitivity Matrix

The raw sensitivity matrix $A_{ji}$ gives the Zernike wavefront change (in microns)
at each WFS position for a unit change in each DOF (microns for translations,
arcseconds for rotations, microns RMS for bending modes).

Our measurement vector is ordered as: [WFS0_z4, WFS1_z4, WFS2_z4, WFS3_z4]
(one Zernike at 4 field positions = 4 measurements, 2 DOFs).

In [ ]:
# Load OFCData and build sensitivity matrix
ofc_data = OFCData('lsst', config_dir=ofc_config_dir)
ofc_data.zn_selected = zn_select

field_angles = [ofc_data.sample_points[sensor] for sensor in sensor_name_list]
print('Field angles (deg):')
for s, fa in zip(sensor_name_list, field_angles):
    print(f'  {s}: ({fa[0]:.4f}, {fa[1]:.4f})')

# Evaluate sensitivity matrix: shape (n_wfs, n_zernike_all, n_dof_all)
dz_sens = SensitivityMatrix(ofc_data)
sens_3d = dz_sens.evaluate(field_angles, rotation_angle=0.0)

# Select our Zernikes and reshape to 2D
zn_idx = zn_select - 4  # Noll -> 0-based
sens_sub = sens_3d[:, zn_idx, :]  # (n_wfs, n_zk, 50)
n_zk = len(zn_select)
n_meas = n_wfs * n_zk

# Reshape: measurement vector = (WFS0_z4, WFS1_z4, ...) x DOFs
A_full = sens_sub.reshape((n_meas, -1))  # (4, 50)

# Select our DOFs
A = A_full[:, dof_indices]  # (4, 2)

print(f'\nRaw sensitivity matrix A (shape {A.shape}):')
print(f'  Rows: {n_meas} measurements ({n_wfs} WFS x {n_zk} Zernike)')
print(f'  Cols: {len(dof_indices)} DOFs')

print(f'\n{"":>10s}', end='')
for name in dof_names:
    print(f'{name:>14s}', end='')
print()
for row_idx in range(n_meas):
    wfs_idx = row_idx // n_zk
    zk_idx = row_idx % n_zk
    label = f'{sensor_name_list[wfs_idx]}_z{zn_select[zk_idx]}'
    print(f'{label:>10s}', end='')
    for col_idx in range(len(dof_indices)):
        print(f'{A[row_idx, col_idx]:>14.6e}', end='')
    print()

print(f'\nUnits: microns of wavefront per unit DOF')
print(f'  Cam_dz unit: microns')
print(f'  Cam_rx unit: arcseconds')

### 1.1 Normalization Weight Components

The normalization has two factors per DOF:
- $r_i$: **range weight** — physical stroke range (microns or arcsec) divided by max actuator force
- $f_i$: **FWHM weight** — RSS of FWHM sensitivity across all field positions

The default OFC normalization is $n_i = r_i \times f_i$.

In [ ]:
# Compute range weights r_i
m1m3_bending_range = ofc_data.m1m3_force_range / 20
m2_bending_range = ofc_data.m2_force_range / 20
m1m3_bmf = BendModeToForce('M1M3', ofc_data)
m2_bmf = BendModeToForce('M2', ofc_data)

range_weights_50 = np.concatenate((
    ofc_data.rb_stroke,
    m1m3_bending_range / np.max(np.abs(m1m3_bmf.rot_mat), axis=0),
    m2_bending_range / np.max(np.abs(m2_bmf.rot_mat), axis=0),
))

# Compute FWHM weights f_i
sensitivity_matrix = dz_sens.evaluate(field_angles, rotation_angle=0.0)
sensitivity_matrix = sensitivity_matrix[:, ofc_data.zn_idx, :]
fwhm_matrix = np.zeros(sensitivity_matrix.shape)
for idy in range(sensitivity_matrix.shape[0]):
    fwhm_matrix[idy, ...] = convertZernikesToPsfWidth(sensitivity_matrix[idy, ...].T).T
fwhm_matrix_2d = fwhm_matrix.reshape((-1, fwhm_matrix.shape[2]))
fwhm_weights_50 = np.zeros(50)
for i in range(50):
    fwhm_weights_50[i] = np.sqrt(np.sum(np.square(fwhm_matrix_2d[:, i])))

# Extract for our DOFs
r_i = range_weights_50[dof_indices]
f_i = fwhm_weights_50[dof_indices]
n_default = ofc_data.normalization_weights[dof_indices]

print('Normalization weight components for selected DOFs:')
print(f'{"DOF":>10s} {"r_i (range)":>14s} {"f_i (FWHM)":>14s} {"r_i * f_i":>14s} {"n_i (stored)":>14s}')
print('-' * 66)
for idx, name in enumerate(dof_names):
    print(f'{name:>10s} {r_i[idx]:>14.6e} {f_i[idx]:>14.6e} {r_i[idx]*f_i[idx]:>14.6e} {n_default[idx]:>14.6e}')

print(f'\nPhysical interpretation:')
for idx, name in enumerate(dof_names):
    print(f'  {name}: range = {r_i[idx]:.4f} (physical stroke), '
          f'FWHM sensitivity = {f_i[idx]:.4f} arcsec/unit')

<a id='norm1'></a>
## 2. Normalization Scheme 1: Default ($\tilde{A} = A \cdot \text{diag}(r_i \times f_i)$)

The current OFC default multiplies each column of $A$ by $n_i = r_i \times f_i$.
This converts the sensitivity matrix from "microns of wavefront per physical DOF unit"
to "microns of wavefront per dimensionless DOF unit", where the dimensionless unit
accounts for both the physical range and the FWHM impact of each DOF.

In [ ]:
# Scheme 1: default n_i = r_i * f_i
n1 = r_i * f_i
A1 = A @ np.diag(n1)

print('Scheme 1: Default normalization n_i = r_i * f_i')
print('=' * 70)
print(f'\nNormalization weights:')
for idx, name in enumerate(dof_names):
    print(f'  n_i({name}) = r_i * f_i = {r_i[idx]:.6e} * {f_i[idx]:.6e} = {n1[idx]:.6e}')

print(f'\nRatio n_i(Cam_dz) / n_i(Cam_rx) = {n1[0] / n1[1]:.4f}')

print(f'\nRaw A (microns wavefront / physical DOF unit):')
print(f'{"":>10s}', end='')
for name in dof_names:
    print(f'{name:>14s}', end='')
print()
for row_idx in range(n_meas):
    wfs_idx = row_idx // n_zk
    label = f'{sensor_name_list[wfs_idx]}_z{zn_select[0]}'
    print(f'{label:>10s}', end='')
    for col_idx in range(len(dof_indices)):
        print(f'{A[row_idx, col_idx]:>14.6e}', end='')
    print()

print(f'\nNormalized Ã₁ = A @ diag(n_i):')
print(f'{"":>10s}', end='')
for name in dof_names:
    print(f'{name:>14s}', end='')
print()
for row_idx in range(n_meas):
    wfs_idx = row_idx // n_zk
    label = f'{sensor_name_list[wfs_idx]}_z{zn_select[0]}'
    print(f'{label:>10s}', end='')
    for col_idx in range(len(dof_indices)):
        print(f'{A1[row_idx, col_idx]:>14.6e}', end='')
    print()

# Column norms (how "important" each DOF looks in the SVD)
col_norms_A = np.linalg.norm(A, axis=0)
col_norms_A1 = np.linalg.norm(A1, axis=0)
print(f'\nColumn norms (||Ã_i||):')
for idx, name in enumerate(dof_names):
    print(f'  {name}: raw = {col_norms_A[idx]:.6e}, normalized = {col_norms_A1[idx]:.6e}')
print(f'  Ratio of column norms (Cam_dz/Cam_rx): '
      f'raw = {col_norms_A[0]/col_norms_A[1]:.4f}, '
      f'normalized = {col_norms_A1[0]/col_norms_A1[1]:.4f}')

<a id='norm2'></a>
## 3. Normalization Scheme 2: FWHM per Normalized DOF

This scheme proceeds in steps:

**Step 1:** Define a "Normalized DOF" (NDOF): $\text{NDOF}_i = \text{DOF}_i / r_i$

This makes all DOFs dimensionless and scaled to their physical range.
The sensitivity matrix becomes $A \cdot \text{diag}(r_i)$: microns of wavefront per unit NDOF.

**Step 2:** Compute FWHM sensitivity per unit NDOF: $\hat{f}_i = f_i \times r_i$

This is the FWHM change (arcsec) for a unit change in NDOF.

**Step 3:** Normalize the sensitivity matrix by dividing each column by $\hat{f}_i$:

$$\tilde{A}_2 = A \cdot \text{diag}(r_i) \cdot \text{diag}(1/\hat{f}_i) = A \cdot \text{diag}(r_i / (f_i \cdot r_i)) = A \cdot \text{diag}(1/f_i)$$

The result has units: microns of wavefront per arcsec of FWHM change.
This is the `inv_f` normalization mode in smatrix_vmode_info.

In [ ]:
# Scheme 2: FWHM per Normalized DOF
print('Scheme 2: FWHM per Normalized DOF')
print('=' * 70)

# Step 1: Sensitivity per Normalized DOF
A_ndof = A @ np.diag(r_i)
print(f'\nStep 1: A_ndof = A @ diag(r_i)  [microns wavefront / unit NDOF]')
print(f'  r_i values:')
for idx, name in enumerate(dof_names):
    print(f'    r_i({name}) = {r_i[idx]:.6e}')

print(f'\n  A_ndof:')
print(f'  {"":>10s}', end='')
for name in dof_names:
    print(f'{name:>14s}', end='')
print()
for row_idx in range(n_meas):
    wfs_idx = row_idx // n_zk
    label = f'{sensor_name_list[wfs_idx]}_z{zn_select[0]}'
    print(f'  {label:>10s}', end='')
    for col_idx in range(len(dof_indices)):
        print(f'{A_ndof[row_idx, col_idx]:>14.6e}', end='')
    print()

# Step 2: FWHM sensitivity per unit NDOF
f_hat = f_i * r_i
print(f'\nStep 2: Normalized FWHM weight f_hat_i = f_i * r_i  [arcsec FWHM / unit NDOF]')
for idx, name in enumerate(dof_names):
    print(f'  f_hat({name}) = {f_i[idx]:.6e} * {r_i[idx]:.6e} = {f_hat[idx]:.6e}')

# Step 3: Normalize by FWHM impact
n2 = 1.0 / f_i  # = r_i / f_hat = r_i / (f_i * r_i) = 1/f_i
A2 = A @ np.diag(n2)

print(f'\nStep 3: Ã₂ = A_ndof @ diag(1/f_hat) = A @ diag(1/f_i)')
print(f'  [microns wavefront / arcsec FWHM change]')
print(f'  n_i = 1/f_i:')
for idx, name in enumerate(dof_names):
    print(f'    n_i({name}) = 1/{f_i[idx]:.6e} = {n2[idx]:.6e}')

print(f'\n  Ã₂:')
print(f'  {"":>10s}', end='')
for name in dof_names:
    print(f'{name:>14s}', end='')
print()
for row_idx in range(n_meas):
    wfs_idx = row_idx // n_zk
    label = f'{sensor_name_list[wfs_idx]}_z{zn_select[0]}'
    print(f'  {label:>10s}', end='')
    for col_idx in range(len(dof_indices)):
        print(f'{A2[row_idx, col_idx]:>14.6e}', end='')
    print()

col_norms_A2 = np.linalg.norm(A2, axis=0)
print(f'\nColumn norms (||Ã₂_i||):')
for idx, name in enumerate(dof_names):
    print(f'  {name}: {col_norms_A2[idx]:.6e}')
print(f'  Ratio (Cam_dz/Cam_rx): {col_norms_A2[0]/col_norms_A2[1]:.4f}')

<a id='svd'></a>
## 4. SVD and V-Mode Comparison

Compare the SVD of the two normalized sensitivity matrices.
In a well-normalized system where two DOFs have equal "importance",
the V-modes should mix them based on their physical degeneracy,
not just because one has a larger column norm.

In [ ]:
# SVD of both schemes
U1, s1, Vh1 = np.linalg.svd(A1, full_matrices=False)
V1 = Vh1.T

U2, s2, Vh2 = np.linalg.svd(A2, full_matrices=False)
V2 = Vh2.T

print('SVD Comparison')
print('=' * 70)

print(f'\nScheme 1 (default: n_i = r_i * f_i):')
print(f'  Singular values: σ = [{", ".join(f"{sv:.6f}" for sv in s1)}]')
print(f'  Condition number: {s1[0]/s1[-1]:.4f}')
print(f'  V matrix (rows = DOFs, cols = V-modes):')
for idx, name in enumerate(dof_names):
    print(f'    {name:>10s}: [{", ".join(f"{V1[idx, m]:>+8.5f}" for m in range(len(s1)))}]')

print(f'\nScheme 2 (FWHM per NDOF: n_i = 1/f_i):')
print(f'  Singular values: σ = [{", ".join(f"{sv:.6f}" for sv in s2)}]')
print(f'  Condition number: {s2[0]/s2[-1]:.4f}')
print(f'  V matrix (rows = DOFs, cols = V-modes):')
for idx, name in enumerate(dof_names):
    print(f'    {name:>10s}: [{", ".join(f"{V2[idx, m]:>+8.5f}" for m in range(len(s2)))}]')

# Also do SVD of raw A for reference
U0, s0, Vh0 = np.linalg.svd(A, full_matrices=False)
V0 = Vh0.T
print(f'\nRaw A (no normalization, for reference):')
print(f'  Singular values: σ = [{", ".join(f"{sv:.6f}" for sv in s0)}]')
print(f'  Condition number: {s0[0]/s0[-1]:.4f}')
print(f'  V matrix (rows = DOFs, cols = V-modes):')
for idx, name in enumerate(dof_names):
    print(f'    {name:>10s}: [{", ".join(f"{V0[idx, m]:>+8.5f}" for m in range(len(s0)))}]')

In [ ]:
# Visual comparison of V-modes
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

schemes = [
    ('Raw A', V0, s0),
    ('Scheme 1: r*f', V1, s1),
    ('Scheme 2: 1/f', V2, s2),
]

for ax, (title, V, sv) in zip(axes, schemes):
    im = ax.imshow(V, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(sv)), [f'm={m+1}\nσ={sv[m]:.3f}' for m in range(len(sv))], fontsize=8)
    ax.set_yticks(range(len(dof_names)), dof_names)
    ax.set_xlabel('V-mode')
    ax.set_ylabel('DOF')
    ax.set_title(title)
    # Annotate values
    for i in range(V.shape[0]):
        for m in range(V.shape[1]):
            ax.text(m, i, f'{V[i,m]:+.3f}', ha='center', va='center', fontsize=10,
                    color='white' if abs(V[i,m]) > 0.5 else 'black')
    fig.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle('V-Mode Composition Under Different Normalizations', fontsize=13)
fig.tight_layout()
plt.show()

<a id='analysis'></a>
## 5. Analysis: Which Normalization Equalizes DOFs?

The goal of normalization is to put all DOFs on the same footing so that the SVD
mixes degenerate DOFs based on their physical effect, not their numerical scale.

**Criterion:** Two DOFs are "on the same footing" in the SVD when equal column norms
in $\tilde{A}$ correspond to equal physical importance. The SVD weights DOFs by their
column norms — a DOF with a larger column norm will dominate the leading V-modes.

We can test this by examining:
1. **Column norm ratio** after normalization — closer to 1.0 means more equal weighting
2. **V-mode mixing** — do the V-modes combine DOFs based on physical degeneracy?
3. **Physical meaning** of the dimensionless unit — what does "1 unit of normalized DOF" mean?

In [ ]:
print('Analysis: Column Norms and DOF Equalization')
print('=' * 70)

print(f'\nColumn norms ||Ã_i|| for each normalization:')
print(f'{"Scheme":>20s}', end='')
for name in dof_names:
    print(f'{name:>14s}', end='')
print(f'{"ratio":>10s}')
print('-' * 58)

for scheme_name, A_norm in [('Raw A', A), ('r*f (default)', A1), ('1/f (inv_f)', A2)]:
    norms = np.linalg.norm(A_norm, axis=0)
    ratio = norms[0] / norms[1]
    print(f'{scheme_name:>20s}', end='')
    for n in norms:
        print(f'{n:>14.6e}', end='')
    print(f'{ratio:>10.4f}')

print(f'\n\nWhat does "1 unit of normalized DOF" mean physically?')
print(f'\nScheme 1 (r*f): 1 unit of dimensionless DOF produces')
for idx, name in enumerate(dof_names):
    phys_per_unit = n1[idx]
    print(f'  {name}: {phys_per_unit:.4e} physical units of change')
    print(f'         -> {f_i[idx] * phys_per_unit:.4e} arcsec FWHM change')

print(f'\nScheme 2 (1/f): 1 unit of normalized DOF produces')
for idx, name in enumerate(dof_names):
    phys_per_unit = n2[idx]
    print(f'  {name}: {phys_per_unit:.4e} physical units of change')
    print(f'         -> {f_i[idx] * phys_per_unit:.4e} arcsec FWHM change')

print(f'\n\nScheme 1: 1 unit of normalized DOF produces FWHM changes:')
fwhm1 = [f_i[idx] * n1[idx] for idx in range(len(dof_names))]
print(f'  {dof_names[0]}: {fwhm1[0]:.4e} arcsec')
print(f'  {dof_names[1]}: {fwhm1[1]:.4e} arcsec')
print(f'  Ratio: {fwhm1[0]/fwhm1[1]:.4f}')

print(f'\nScheme 2: 1 unit of normalized DOF produces FWHM changes:')
fwhm2 = [f_i[idx] * n2[idx] for idx in range(len(dof_names))]
print(f'  {dof_names[0]}: {fwhm2[0]:.4e} arcsec')
print(f'  {dof_names[1]}: {fwhm2[1]:.4e} arcsec')
print(f'  Ratio: {fwhm2[0]/fwhm2[1]:.4f}')

print(f'\n\nConclusion:')
print(f'  Scheme 1 (r*f): each DOF at "1 unit" produces FWHM proportional to r_i*f_i^2.')
print(f'    DOFs with large range AND high FWHM sensitivity dominate.')
print(f'  Scheme 2 (1/f): each DOF at "1 unit" produces the SAME Zernike-per-arcsec-FWHM.')
print(f'    The SVD sees DOFs as having equal importance per unit FWHM impact.')
print(f'    This puts DOFs "on the same footing" with respect to image quality.')